# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item's activity for March 2026 (aggregated
from the daily fact table's report_date × client × content grain, restricted to
month=2026-03 — a mid-panel month, not the sealed final-month sample).

Time window: I split March in half — days 1–15 (the "known" half, my features) and
days 16–31 (the "outcome" half, my label). This keeps both feature and label inside
the same partition while still respecting past→future ordering.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb

import os
from google.colab import userdata
HF_TOKEN = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature  — imp_first_half, clicks_first_half, avg_position_first_half
           (from fact_content_daily_performance, days 1–15 March):
           knowable at the decision moment (end of the first half) because
           GSC logs impressions/clicks/position daily, with no lag into the future.
Feature  — content_age_days, days_since_last_update (from dim_content):
           static editorial metadata, knowable at any point, never depends on
           an outcome window.
Label    — is_declining = imp_second_half < 0.8 * imp_first_half
           (days 16–31 vs days 1–15). This is the proxy I'm predicting, never a feature.
Context  — client_hash_id, content_hash_id: used only for grouping/joining/
           client-holdout splits, never fed to a model.
Excluded — ga4_data_available = FALSE rows: GA4 columns are zero-filled (not
           genuinely zero) before a client's GA4 start date, so including them
           unfiltered would silently corrupt any engagement feature. Excluded
           for this pass rather than risk misreading a zero-fill as "no engagement."

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>10,} rows")


dim_clients         104 rows
dim_content     519,606 rows
fact_march    9,841,378 rows


In [18]:
con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [19]:
data = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN f.report_date >  DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_clicks ELSE 0 END)      AS clicks_first_half,
            AVG(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END)        AS avg_position_first_half
        FROM {TABLES['fact_march']} f
        GROUP BY 1, 2
        HAVING imp_first_half >= 50
    ),
    content_agg AS (
        SELECT
            content_hash_id,
            ANY_VALUE(content_created_date) AS content_created_date,
            ANY_VALUE(content_updated_date) AS content_updated_date
        FROM {TABLES['dim_content']}
        GROUP BY content_hash_id
    )
    SELECT
        d.*,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-15') AS content_age_days,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-15') AS days_since_last_update
    FROM daily d
    LEFT JOIN content_agg c USING (content_hash_id)
""").df()

data["is_declining"] = (data["imp_second_half"] < 0.8 * data["imp_first_half"]).astype(int)
print(f"{len(data):,} content items with enough March-1st-half volume")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 content items with enough March-1st-half volume


,client_hash_id,content_hash_id,imp_first_half,imp_second_half,clicks_first_half,avg_position_first_half,content_age_days,days_since_last_update,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,2350.0,6.0,6.327311,380,-113,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,208.0,0.0,3.906852,380,-64,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,1925.0,3.0,6.473735,380,-66,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,2504.0,8.0,7.259861,380,-113,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,189.0,1.0,3.860842,380,-64,1


In [20]:
data["days_since_last_update"] = data["days_since_last_update"].clip(lower=0)
print("Negative values before clip:", (data["days_since_last_update"] < 0).sum())
data[["content_age_days", "days_since_last_update"]].describe()

Negative values before clip: 0


,content_age_days,days_since_last_update
count,92548.000000,92548.000000
mean,182.381996,3.601688
std,124.086678,8.530335
min,1.000000,0.000000
25%,62.000000,0.000000
50%,177.000000,0.000000
75%,250.000000,0.000000
max,478.000000,248.000000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries, run below:
1. Grain: no (client, content) pair has more than one row in the March partition
   after daily aggregation.
2. Row count + date span: confirms March partition covers exactly 2026-03-01 to
   2026-03-31.
3. Availability: how many rows survive an IS TRUE filter on ga4_data_available.

Then: five features (each with an "available when" line), and the deliberate
leakage trap.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 — grain probe
grain_check = con.sql(f"""
    SELECT COUNT(*) AS n_violations
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
        FROM {TABLES['fact_march']}
        GROUP BY 1, 2, 3
        HAVING c > 1
    )
""").df()
print(grain_check)
# Query 2 — row count + date span
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS start_d, MAX(report_date) AS end_d
    FROM {TABLES['fact_march']}
""").df()
print(span)

# Query 3 — availability, filtered IS TRUE
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_march']}
""").df()
avail["pct_available"] = (avail["ga4_available_rows"] / avail["total_rows"] * 100).round(1)
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_violations
0             0
    n_rows    start_d      end_d
0  9841378 2026-03-01 2026-03-31
   total_rows  ga4_available_rows  pct_available
0     9841378            413966.0            4.2


In [22]:
print("""
Five features and why each is knowable at the decision moment (end of March 15):

1. imp_first_half           — daily GSC impressions already logged by day 15.
2. clicks_first_half        — same: GSC logs clicks same-day, no future lag.
3. avg_position_first_half  — GSC position is measured per day, not retroactively revised.
4. content_age_days         — computed relative to March 15 itself, so it's exactly
                               what would have been known at that moment, not today.
5. days_since_last_update   — same: computed relative to March 15, clipped so it can
                               never reflect an update that happened after the decision point.
""")

feature_cols = ["imp_first_half", "clicks_first_half", "avg_position_first_half",
                "content_age_days", "days_since_last_update"]
model_frame = data.dropna(subset=feature_cols + ["is_declining"])
print(f"{len(model_frame):,} rows with complete features")


Five features and why each is knowable at the decision moment (end of March 15):

1. imp_first_half           — daily GSC impressions already logged by day 15.
2. clicks_first_half        — same: GSC logs clicks same-day, no future lag.
3. avg_position_first_half  — GSC position is measured per day, not retroactively revised.
4. content_age_days         — computed relative to March 15 itself, so it's exactly
                               what would have been known at that moment, not today.
5. days_since_last_update   — same: computed relative to March 15, clipped so it can
                               never reflect an update that happened after the decision point.

92,548 rows with complete features


In [23]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def quick_score(X, y):
    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    tree.fit(X, y)
    return tree.score(X, y)

X_honest = model_frame[feature_cols]
y = model_frame["is_declining"]
print(f"Honest score (5 real features): {quick_score(X_honest, y):.3f}")

# --- THE TRAP: add a label-derived column on purpose ---
model_frame["leaky_ratio"] = model_frame["imp_second_half"] / model_frame["imp_first_half"]
X_leaky = model_frame[feature_cols + ["leaky_ratio"]]
print(f"Leaky score (adding imp_second_half-derived ratio): {quick_score(X_leaky, y):.3f}")
print("-> leaky_ratio is built directly from the label's own definition. Deleting it:")

model_frame = model_frame.drop(columns=["leaky_ratio"])
print(f"Final honest score kept for the record: {quick_score(X_honest, y):.3f}")

Honest score (5 real features): 0.542
Leaky score (adding imp_second_half-derived ratio): 1.000
-> leaky_ratio is built directly from the label's own definition. Deleting it:
Final honest score kept for the record: 0.542


In [24]:
print(f"\nFor reference, majority-class base rate: {max(y.mean(), 1 - y.mean()):.3f}")
print("The honest score barely beats this — these 5 features alone aren't strongly predictive yet.")


For reference, majority-class base rate: 0.714
The honest score barely beats this — these 5 features alone aren't strongly predictive yet.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: this analysis used one mid-panel month (March 2026) split into
two 15/16-day halves, and excluded all GA4-unavailable rows (95.8% of March rows
have no GA4 data at all, per Query 3 — most clients in this warehouse are GSC-only
or have no access). That means any GA4-based signal (sessions, engagement,
pageviews) would only be usable for a small, non-random subset of clients, and
this analysis as built doesn't even touch GA4 columns because of that.

A second limitation surfaced directly while building this: dim_content's
content_updated_date can fall after the decision date for some rows (my
days_since_last_update was negative before I clipped it to 0). That means the
"last updated" field isn't a clean historical record — it can reflect edits made
after the point I'm trying to predict from, so clipping hides the leak rather than
eliminating the underlying data-quality issue. A future version should exclude
rather than clip content whose recorded update postdates the decision point,
since clipping to 0 quietly treats "updated in the future" the same as "updated
today," which isn't true.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Back up the stated limitations with numbers
ga4_unavailable_pct = 100 - avail["pct_available"].iloc[0]
print(f"GA4 unavailable in {ga4_unavailable_pct:.1f}% of March rows — confirms GA4 signals aren't usable for most of this data.")

# Confirm how many dim_content rows had an update date after the decision point (before clipping)
future_updates = con.sql(f"""
    SELECT COUNT(*) AS n_future_updates
    FROM {TABLES['dim_content']}
    WHERE content_updated_date > DATE '2026-03-15'
""").df()
print(future_updates)

GA4 unavailable in 95.8% of March rows — confirms GA4 signals aren't usable for most of this data.
   n_future_updates
0            382794


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.